# Notebook 5: Causal Analysis

## 5.0 What This Notebook Is About

Imagine you are testing three different camera setups for a self-driving car:
a single front-facing camera, three cameras pointing in different directions,
and a laser rangefinder (lidar) that draws a bird's-eye-view map of nearby
obstacles. We trained a separate AI driver for each setup and ran thousands
of test drives inside a driving simulator called CARLA.

The obvious question is: *which setup drives the farthest?* But the obvious
comparison is also misleading. What if the lidar car happened to get tested
more often in good weather, while the single-camera car got stuck with rain
and night runs? A straight average would make lidar look better even if the
sensor had nothing to do with it.

**Causal inference** is a set of techniques that let us isolate the effect
of one thing — here, the sensor choice — while holding everything else
(weather, town, driving style) equal. Think of it like a controlled clinical
trial, but for self-driving cars.

This notebook answers **nine specific causal questions** in three groups:

---

### Group 1 — Which sensor makes the final AI drive the farthest?
*We compare the fully trained (PPO) models only, after adjusting for weather and town.*

| # | Question | Method |
|---|---|---|
| Q1 | Does the three-camera setup **cause** higher route completion than a single camera? | Propensity Score Matching + sensitivity test |
| Q2 | Does lidar **cause** higher route completion than a single camera? | Propensity Score Matching + sensitivity test |
| Q3 | Does lidar **cause** higher route completion than the three-camera setup? | Propensity Score Matching + sensitivity test |

### Group 2 — How much do tough conditions hurt the AI?
*We compare performance in hard conditions vs. an easy baseline.*

| # | Question | Method |
|---|---|---|
| Q4 | Does driving in hard rain **cause** lower route completion than clear noon sun? | Propensity Score Matching |
| Q5 | Does driving at night **cause** lower route completion than clear noon sun? | Propensity Score Matching |
| Q6 | Does the "hurry" driving style **cause** lower route completion than the "chill" style? | Propensity Score Matching |

### Group 3 — Does the sensor affect how well the AI learns from practice?
*We compare the improvement each sensor got from RL fine-tuning (PPO), relative to its starting point (BC).*

| # | Question | Method |
|---|---|---|
| Q7 | Did PPO practice improve the three-camera AI **more** than the single-camera AI? | Difference-in-Differences |
| Q8 | Did PPO practice improve the lidar AI **more** than the single-camera AI? | Difference-in-Differences |
| Q9 | Did PPO practice improve the lidar AI **more** than the three-camera AI? | Difference-in-Differences |

---

**Q1–Q3** are the core claims of this project.
**Q4–Q6** check how robust the AI is to real-world conditions.
**Q7–Q9** ask whether some sensors help the AI *learn faster* — not just perform better.

In [ ]:
import sys
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import linear_sum_assignment

# Ensure the project root is on sys.path for src imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.agents.causal_agent import CausalAnalysisAgent

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 5.1 Configuration

### 5.1.1 Loading Parameters

All the settings that control this analysis live in a single file: `configs/causal.yaml`.
Think of it like a recipe card — instead of baking the numbers directly into the notebook,
we read them from the card so every script that runs this analysis uses the exact same values.

Three settings matter most:

- **Bootstrap resamples (1,000 by default):** To build a confidence interval, we repeat
  the analysis 1,000 times, each time drawing a slightly different random sample of test
  drives. The spread of those 1,000 results tells us how much the answer would change if
  we had collected slightly different data. More resamples → narrower, more trustworthy intervals.

- **Minimum treated episodes (20 by default):** If a comparison group has fewer than 20
  test drives, we skip that treatment entirely. A handful of data points can produce a
  wildly misleading average, so this threshold protects against noise masquerading as a
  real effect.

- **Random seed (42):** Setting a fixed starting number for all random operations means
  anyone who re-runs this notebook will get bit-for-bit identical results. Without it,
  the bootstrap draws would differ every run.

In [ ]:
cfg = load_config("causal")

print(f"Bootstrap resamples      : {cfg['n_bootstrap']}")
print(f"Min treated units        : {cfg['min_treated']}")
print(f"Random seed              : {cfg['random_seed']}")
print(f"PPO-only filter (PSM)    : {cfg['ppo_only']}")
print(f"Primary treatments (Q1–Q3) : {len(cfg['primary_treatments'])}")
print(f"Secondary treatments (Q4–Q6): {len(cfg['secondary_treatments'])}")
print(f"DiD comparisons (Q7–Q9)  : {len(cfg['did_comparisons'])}")

### 5.1.2 Treatment Definitions

In a clinical trial you have a *treatment group* (patients who get the new drug) and a
*control group* (patients who get the existing drug or a placebo). We do the same thing
with test drives.

For **Q1–Q6** we use only the fully trained (PPO) drives — because BC is an intermediate
checkpoint, not the final policy, and we don't want it to dilute the comparison.
For **Q7–Q9** we deliberately use *both* BC and PPO drives so we can measure how much
each sensor setup improved from RL training.

---

**Questions 1–3 — which sensor drives farthest? (with robustness check)**

| Q | Treatment name | "Treated" car | "Control" car | What we hold equal |
|---|---|---|---|---|
| Q1 | sensor_single_vs_multi | Three-camera | Single-camera | Weather, town |
| Q2 | sensor_single_vs_lidar | Lidar | Single-camera | Weather, town |
| Q3 | sensor_multi_vs_lidar | Lidar | Three-camera | Weather, town |

**Questions 4–6 — how much do tough conditions hurt?**

| Q | Treatment name | "Treated" condition | "Control" condition | What we hold equal |
|---|---|---|---|---|
| Q4 | rain | Hard rain at noon | Clear sky at noon | Sensor type, town |
| Q5 | night | Clear sky at night | Clear sky at noon | Sensor type, town |
| Q6 | style_hurry_vs_chill | Hurry driving style | Chill driving style | Weather, sensor type, town |

**Questions 7–9 — does sensor type affect how well the AI learns? (DiD)**

No matching needed here. The sensor setup is fixed by design for each trained model,
so the only thing that changes between the BC and PPO rows is the training step.

| Q | Comparison | What the DiD number measures |
|---|---|---|
| Q7 | Three-camera vs single-camera | Did three-camera improve *more* from RL practice? |
| Q8 | Lidar vs single-camera | Did lidar improve *more* from RL practice? |
| Q9 | Lidar vs three-camera | Did lidar improve *more* than three-camera from RL practice? |

## 5.2 Causal Framework

*Why a straight average comparison isn't enough — and the tools we use to fix it.*

### 5.2.1 Research Questions at a Glance

Each question maps to a single number — either an **ATE** (Average Treatment Effect, Q1–Q6)
or a **DiD** (Difference-in-Differences, Q7–Q9). The tables below show what it means when
that number is positive, negative, or near zero.

---

**Q1–Q3: Which sensor makes the AI drive the farthest?**

| Q | Question | Positive ATE means… | Negative ATE means… |
|---|---|---|---|
| Q1 | Does three-camera cause higher route completion than single-camera? | Three-camera drives farther | Single-camera is sufficient — adding cameras doesn't help |
| Q2 | Does lidar cause higher route completion than single-camera? | Lidar drives farther | Single-camera is competitive with lidar |
| Q3 | Does lidar cause higher route completion than three-camera? | Lidar outperforms three-camera | Three-camera matches or beats lidar |

---

**Q4–Q6: How much do tough conditions hurt the AI?**

| Q | Question | Positive ATE means… | Negative ATE means… |
|---|---|---|---|
| Q4 | Does hard rain cause lower route completion than clear noon? | Rain somehow *helps* — likely a data issue, check overlap | Rain hurts performance |
| Q5 | Does night driving cause lower route completion than clear noon? | Night somehow *helps* — likely a data issue | Darkness hurts performance |
| Q6 | Does the "hurry" style cause lower route completion than "chill"? | Hurry style completes more route | Aggressive driving increases crashes/failures |

---

**Q7–Q9: Does the sensor affect how well the AI learns from practice?**

| Q | Question | Positive DiD means… | Near-zero DiD means… |
|---|---|---|---|
| Q7 | Did PPO benefit three-camera more than single-camera? | Three-camera's wider view helps the AI learn faster | Both sensors benefit equally from practice |
| Q8 | Did PPO benefit lidar more than single-camera? | Lidar's bird's-eye map helps the AI learn faster | Improvement is independent of sensor type |
| Q9 | Did PPO benefit lidar more than three-camera? | Lidar has a learning advantage over three-camera | Both benefit equally from practice |

---

**The robustness check (Q1–Q3 only) — the Rosenbaum Γ:**
After finding an answer for Q1–Q3, we ask: *how strong would a hidden confound need to
be before it could explain away our result?* This is reported as **Γ** (gamma).
A Γ of 2.0 means: even if an unobserved factor made one group twice as likely to be
assigned to that sensor type, our conclusion would still hold. Higher Γ = harder to
explain away = more trustworthy finding.

### 5.2.2 The Fine Print — Assumptions We Are Making

Every statistical method comes with conditions that must be roughly true for the results
to be trustworthy. Here are the six core assumptions, written as plainly as possible,
with a note on how believable each one is in a driving simulator.

---

**1. Each test drive is its own story.**
One drive's outcome is not influenced by what happened in any other drive.
*(Plausible — CARLA episodes are independent runs with fresh spawns.)*

**2. Any route could have used any sensor or weather.**
For every combination of town and weather, there exist both treated and control drives.
If some conditions only ever appear in one sensor group, we can't make a fair comparison.
*(Partially checked — the overlap diagnostic in §5.6 flags violations.)*

**3. We measured all the important background variables.**
After matching on weather and town, treatment assignment is effectively random.
Weather and town are the main factors that could unfairly favor one group. Spawn-point
geometry and NPC behavior are *not* controlled — that's the biggest remaining risk.
*(Partially verified — Rosenbaum Γ bounds how badly an unobserved factor would need to act.)*

**4. The matching model is roughly correct.**
We use a logistic regression to predict which drives ended up in each group. If the
true assignment mechanism is highly nonlinear, the model will be off and matching
quality will suffer.
*(Reasonable for categorical covariates like weather and town codes.)*

**5. After matching, the groups are actually balanced.**
We verify this numerically with the SMD (Standardized Mean Difference) in §5.3.
If any covariate still has SMD > 0.1 after matching, we flag that treatment's result.
*(Verified — see post-matching balance table in §5.4.2.)*

**6. For Q7–Q9 only: both sensors would have improved equally without PPO.**
This "parallel trends" assumption says the BC→PPO improvement gap we observe is *caused*
by the sensor, not by a pre-existing advantage one sensor already had. It cannot be
proven from the data — it is a judgment call about the training setup.
*(Plausible — BC training procedure is identical across all three sensors.)*

### 5.2.3 What Number Are We Actually Estimating?

The core number this analysis produces is called the **Average Treatment Effect (ATE)**.
In plain language: *on average, how much farther does a car drive when it uses the treated
setup instead of the control setup?*

The formal definition uses the idea of a *potential outcome* — the route completion a drive
*would have gotten* under a counterfactual scenario. We write $Y(1)$ for "what would happen
under treatment" and $Y(0)$ for "what would happen under control":

$$\text{ATE} = \mathbb{E}[Y(1) - Y(0)]$$

Of course we can never observe both outcomes for the same drive at the same time — a car
can't simultaneously have lidar and not have lidar. Propensity score matching sidesteps
this by pairing each treated drive with the most similar control drive, and using the
control drive as a stand-in for the counterfactual.

The sample estimate we actually compute is:

$$\text{ATE} = \frac{1}{n}\sum_{i=1}^{n}\bigl(Y_i^{\text{treated}} - Y_i^{\text{matched control}}\bigr)$$

where $n$ is the number of matched pairs. Route completion ranges from 0 (the car crashed
immediately) to 1 (the car finished the full route), so the ATE is in percentage-point
terms. For example, **ATE = +0.08** means the treated setup completed an extra **8 percentage
points** of route on average — say, 75% vs. 67% of a 1 km route.

## 5.3 Pre-Matching Diagnostics

*Checking whether the treated and control groups started out similar before we touch the data.*

### 5.3.1 Are the Groups Similar Before Matching?

Before doing any matching, we check whether the treated and control groups are *already*
comparable on the covariates we plan to control (weather, town, sensor type).
If they're very similar to begin with, matching will be easy. If they're very different,
matching will have to work harder — and we'll want to double-check afterward that it succeeded.

The measuring stick we use is the **Standardized Mean Difference (SMD)**:

$$\text{SMD} = \frac{|\bar{x}_{\text{treated}} - \bar{x}_{\text{control}}|}{\sqrt{(\sigma_{\text{treated}}^2+\sigma_{\text{control}}^2)/2}}$$

Think of it as: *how big is the gap between the groups' averages, measured in units of their
combined spread?* An SMD of **0** means perfect balance. An SMD above **0.1** is the
conventional warning sign that imbalance could bias the result. Values above 0.25 are a red flag.

The table below shows the pre-matching SMD for each treatment–covariate pair.
High values here *motivate* the matching step — they should drop below 0.1 after matching.

In [ ]:
results_dir = PROJECT_ROOT / "results"
eval_path = results_dir / "eval_results.json"

with open(eval_path) as f:
    records = json.load(f)

# Apply PPO-only filter to match what the agent does
records_ppo = [r for r in records if r.get("model_type") == "ppo"]
print(f"Loaded {len(records)} total records; using {len(records_ppo)} PPO records.")

# Encode covariates
weather_enc = LabelEncoder()
town_enc = LabelEncoder()
suite_enc = LabelEncoder()
weather_enc.fit([r["weather"] for r in records_ppo])
town_enc.fit([r["town"] for r in records_ppo])
suite_enc.fit([r.get("sensor_suite", "single_cam") for r in records_ppo])

cov_arrays = {
    "weather_code": weather_enc.transform(
        [r["weather"] for r in records_ppo]).astype(float),
    "town_code": town_enc.transform(
        [r["town"] for r in records_ppo]).astype(float),
    "sensor_suite_code": suite_enc.transform(
        [r.get("sensor_suite", "single_cam") for r in records_ppo]).astype(float),
}


def compute_smd(x_treated, x_control):
    pooled_std = np.sqrt((x_treated.var() + x_control.var()) / 2)
    if pooled_std == 0:
        return 0.0
    return float(abs(x_treated.mean() - x_control.mean()) / pooled_std)


def build_filter(condition):
    if isinstance(condition, list):
        filters = [build_filter(c) for c in condition]
        return lambda r: all(f(r) for f in filters)
    field, op, value = condition["field"], condition["op"], condition["value"]
    if op == "eq":
        return lambda r, _f=field, _v=value: r.get(_f) == _v
    elif op == "in":
        _vset = set(value)
        return lambda r, _f=field, _vs=_vset: r.get(_f) in _vs
    raise ValueError(f"Unknown operator: {op}")


# Combine primary and secondary treatments for diagnostics
all_treatments = cfg["primary_treatments"] + cfg["secondary_treatments"]
smd_rows = []
for tdef in all_treatments:
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = [i for i, r in enumerate(records_ppo) if t_filter(r)]
    c_idx = [i for i, r in enumerate(records_ppo) if c_filter(r)]
    for cov_name in tdef["covariates"]:
        if cov_name not in cov_arrays:
            continue
        arr = cov_arrays[cov_name]
        smd_val = (
            compute_smd(arr[t_idx], arr[c_idx])
            if len(t_idx) > 0 and len(c_idx) > 0 else float("nan")
        )
        smd_rows.append({
            "treatment": tdef["name"],
            "covariate": cov_name,
            "n_treated": len(t_idx),
            "n_control": len(c_idx),
            "SMD_pre_match": round(smd_val, 4),
        })

smd_df = pd.DataFrame(smd_rows)
print("\nPre-matching covariate balance (SMD):")
display(smd_df)

### 5.3.2 Propensity Scores — Who "Looks Like" They Should Be in the Treated Group?

A **propensity score** is the probability that a given test drive *would be assigned* to the
treated group, based on its observed covariates (weather and town):

$$e(x) = P(T = 1 \mid X = x)$$

We estimate this with a logistic regression. Once we have a propensity score for every drive,
we can match each treated drive to the control drive with the closest score — meaning the
control drive that was in the most similar weather-and-town combination.

**Why plot the distributions?** If the propensity score distributions of the treated and control
groups overlap well, matching will find good pairs across the full range. If they barely
overlap — for example, all lidar drives happened in clear weather while all single-camera
drives happened in rain — then matching can only pair a handful of drives, and the ATE
estimate will be unreliable.

In the plots below you want the **blue (treated)** and **red (control)** histograms to overlap
in the middle, not hug opposite ends. Heavy overlap = healthy comparison.

In [ ]:
n_treatments = len(all_treatments)
n_cols = 3
n_rows = (n_treatments + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for idx, tdef in enumerate(all_treatments):
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = np.array([i for i, r in enumerate(records_ppo) if t_filter(r)])
    c_idx = np.array([i for i, r in enumerate(records_ppo) if c_filter(r)])

    if len(t_idx) < cfg["min_treated"] or len(c_idx) < cfg["min_treated"]:
        axes[idx].set_title(f"{tdef['name']} -- insufficient data")
        continue

    all_idx = np.concatenate([t_idx, c_idx])
    T = np.array([1] * len(t_idx) + [0] * len(c_idx))
    X = np.column_stack([
        cov_arrays[cov][all_idx]
        for cov in tdef["covariates"]
        if cov in cov_arrays
    ])

    lr = LogisticRegression(C=1.0, max_iter=500, random_state=cfg["random_seed"])
    lr.fit(X, T)
    pscore = lr.predict_proba(X)[:, 1]

    ax = axes[idx]
    ax.hist(pscore[T == 1], bins=20, alpha=0.6, label="Treated", color="steelblue")
    ax.hist(pscore[T == 0], bins=20, alpha=0.6, label="Control", color="coral")
    is_primary = tdef.get("rosenbaum", False)
    ax.set_title(f"{tdef['name']} ({'primary' if is_primary else 'secondary'})")
    ax.set_xlabel("Propensity Score")
    ax.legend(fontsize=8)

for j in range(n_treatments, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Pre-Matching Propensity Score Distributions (PPO records only)", fontsize=13)
fig.tight_layout()
plt.show()

## 5.4 Propensity Score Matching

*Pairing each treated drive with its closest comparable control drive, then computing the effect.*

### 5.4.1 Running the Full Analysis (PSM + DiD)

We now hand the work to `CausalAnalysisAgent`, which runs both pipelines back-to-back.

**PSM pipeline (Q1–Q6) — step by step:**
1. Fit a logistic regression to predict which drives ended up in the treated vs. control group.
2. For each treated drive, find the control drive with the most similar propensity score —
   without reusing any control drive twice (optimal 1-to-1 matching via `linear_sum_assignment`).
3. Compute the ATE: average the route-completion gap across all matched pairs.
4. Run 1,000 bootstrap resamples to build a 95% confidence interval.
5. For Q1–Q3, compute the Rosenbaum Γ sensitivity bound.

**DiD pipeline (Q7–Q9) — the formula:**

$$\text{DiD} = \bigl(\bar{Y}^{\text{PPO}}_{\text{treated}} - \bar{Y}^{\text{BC}}_{\text{treated}}\bigr) - \bigl(\bar{Y}^{\text{PPO}}_{\text{control}} - \bar{Y}^{\text{BC}}_{\text{control}}\bigr)$$

No matching needed — uses all BC and PPO records directly. Bootstrap CI is still computed.

Results are saved to `results/causal_results.json` as `{"psm": [...], "did": [...]}`.  
`psm` answers Q1–Q6; `did` answers Q7–Q9.

In [ ]:
agent = CausalAnalysisAgent(
    results_dir=str(results_dir),
    plots_dir=str(results_dir / "causal_plots"),
)
causal_output = agent.run()
psm_results = causal_output["psm"]
did_results = causal_output["did"]
print(f"PSM treatments analysed : {len(psm_results)} (Q1–Q6)")
print(f"DiD comparisons analysed: {len(did_results)} (Q7–Q9)")

### 5.4.2 Did the Matching Actually Work?

Matching is only useful if it achieved balance. After matching, we re-compute the SMD
for every covariate. Our goal: **every SMD should fall below 0.1** after matching.

If a covariate is still imbalanced (SMD ≥ 0.1) after matching, the matched pairs for
that treatment are not truly comparable on that variable, and the ATE estimate for
that treatment should be treated with skepticism.

The table below shows SMD before and after matching, side by side.
A **"Yes"** in the "Balanced?" column means the covariate passed the 0.1 threshold.
You want every row to say Yes before drawing conclusions from that treatment.

In [ ]:
rng = np.random.default_rng(cfg["random_seed"])
post_smd_rows = []

for tdef in all_treatments:
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = np.array([i for i, r in enumerate(records_ppo) if t_filter(r)])
    c_idx = np.array([i for i, r in enumerate(records_ppo) if c_filter(r)])

    if len(t_idx) < cfg["min_treated"] or len(c_idx) < cfg["min_treated"]:
        continue

    all_idx = np.concatenate([t_idx, c_idx])
    T = np.array([1] * len(t_idx) + [0] * len(c_idx))
    X = np.column_stack([
        cov_arrays[cov][all_idx]
        for cov in tdef["covariates"]
        if cov in cov_arrays
    ])

    lr = LogisticRegression(C=1.0, max_iter=500, random_state=cfg["random_seed"])
    lr.fit(X, T)
    pscore = lr.predict_proba(X)[:, 1]

    # Optimal 1:1 matching without replacement (mirrors CausalAnalysisAgent)
    pscore_t = pscore[T == 1]
    pscore_c = pscore[T == 0]
    cost = np.abs(pscore_t[:, None] - pscore_c[None, :])
    row_ind, col_ind = linear_sum_assignment(cost)

    treated_positions = np.where(T == 1)[0][row_ind]
    control_positions = np.where(T == 0)[0][col_ind]

    for cov_name in tdef["covariates"]:
        if cov_name not in cov_arrays:
            continue
        arr = cov_arrays[cov_name]
        cov_t = arr[all_idx[treated_positions]]
        cov_m = arr[all_idx[control_positions]]
        smd_val = compute_smd(cov_t, cov_m)
        post_smd_rows.append({
            "treatment": tdef["name"],
            "covariate": cov_name,
            "SMD_post_match": round(smd_val, 4),
            "balanced": "Yes" if smd_val < 0.1 else "No",
        })

post_smd_df = pd.DataFrame(post_smd_rows)
print("Post-matching covariate balance (SMD):")
display(post_smd_df)

comparison = smd_df.merge(post_smd_df, on=["treatment", "covariate"], how="inner")
print("\nPre vs Post matching SMD:")
display(comparison[["treatment", "covariate", "SMD_pre_match", "SMD_post_match", "balanced"]])

## 5.5 Results

*What the numbers say — translated into plain English.*

### 5.5.1 Main Results — Average Treatment Effects (Q1–Q6)

The table below is the core output of the PSM analysis. Here is how to read each column:

| Column | What it means |
|---|---|
| **Q** | Which research question this row answers |
| **ATE** | How many percentage points of route completion the treated setup gained vs. control (positive = treated is better; negative = control is better) |
| **95% CI** | The range we are 95% confident the true ATE falls within. If this range does **not** include zero, the result is statistically significant. |
| **Sig.** | "Yes" if the confidence interval excludes zero |
| **Rosenbaum Γ** | How strong a hidden confound would need to be to flip the result (Q1–Q3 only — higher is better; ≥ 1.5 is considered robust) |
| **n_treated** | Number of treated drives used after the min-treated threshold |
| **overlap_ok** | Whether propensity score overlap was adequate for valid matching |

**Example:** ATE = +0.10 means the treated car completed **10 percentage points more** of the route on average — for instance, finishing 80% vs. 70% of a 1 km route.

In [ ]:
causal_path = results_dir / "causal_results.json"
with open(causal_path) as f:
    causal_results_loaded = json.load(f)

psm_loaded = causal_results_loaded["psm"]

q_labels = {
    "sensor_single_vs_multi": "Q1",
    "sensor_single_vs_lidar": "Q2",
    "sensor_multi_vs_lidar":  "Q3",
    "rain":                   "Q4",
    "night":                  "Q5",
    "style_hurry_vs_chill":   "Q6",
}

ate_rows = []
for r in psm_loaded:
    gamma = r.get("rosenbaum_gamma")
    ate_rows.append({
        "Q": q_labels.get(r["treatment"], "—"),
        "Treatment": r["treatment"],
        "Tier": "primary" if r.get("is_primary") else "secondary",
        "ATE": f"{r['ate']:+.4f}",
        "95% CI": f"[{r['ci_lower']:+.4f}, {r['ci_upper']:+.4f}]",
        "Sig.": "Yes" if r["ci_lower"] > 0 or r["ci_upper"] < 0 else "No",
        "Rosenbaum Γ": f"{gamma:.2f}" if gamma is not None else "n/a",
        "n_treated": r["n_treated"],
        "overlap_ok": r["overlap_ok"],
    })

ate_df = pd.DataFrame(ate_rows)
print("PSM Results — Q1 through Q6 (PPO records only):")
display(ate_df)

### 5.5.2 What Do the Results Tell Us?

The code below walks through each of Q1–Q6 and prints a plain-English summary: whether
the effect was significant, which direction it points, and — for Q1–Q3 — how much hidden
confounding it could survive.

Treat any result where `overlap_ok = False` as preliminary: it means there weren't enough
comparable control drives to make a fully trustworthy comparison.

In [ ]:
q_labels_inv = {v: k for k, v in q_labels.items()}

for r in psm_loaded:
    name = r["treatment"]
    q = q_labels.get(name, "?")
    ate = r["ate"]
    ci_lo, ci_hi = r["ci_lower"], r["ci_upper"]
    gamma = r.get("rosenbaum_gamma")
    sig = ci_lo > 0 or ci_hi < 0
    direction = "increases" if ate > 0 else "decreases"
    robust = (
        "robust to moderate hidden bias" if gamma and gamma >= 1.5
        else "sensitive to small unmeasured confounders" if gamma and gamma <= 1.1
        else "moderately robust to hidden bias"
    ) if gamma else "n/a (secondary treatment)"
    overlap = "adequate" if r["overlap_ok"] else "POOR — interpret with caution"

    sig_str = "statistically significant (CI excludes zero)" if sig else "not statistically significant"
    print(f"{q} | {name}")
    print(f"  ATE = {ate:+.4f}  95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]  — {sig_str}")
    print(f"  Treatment {direction} route completion by {abs(ate)*100:.1f} pp on average.")
    if gamma:
        print(f"  Rosenbaum Γ = {gamma:.2f} — {robust}.")
    print(f"  Overlap: {overlap}")
    print()

## 5.5b Did the Sensor Affect How Fast the AI Learned? (Q7–Q9)

These three questions don't ask *which sensor drives farthest*. They ask a different thing:
was the **jump from imitation learning (BC) to reinforcement learning (PPO)** bigger for
some sensors than others?

**The formula:**

$$\text{DiD} = \underbrace{\bigl(\bar{Y}^{\text{PPO}}_{\text{treated}} - \bar{Y}^{\text{BC}}_{\text{treated}}\bigr)}_{\text{treated suite's BC→PPO improvement}} \;-\; \underbrace{\bigl(\bar{Y}^{\text{PPO}}_{\text{control}} - \bar{Y}^{\text{BC}}_{\text{control}}\bigr)}_{\text{control suite's BC→PPO improvement}}$$

**How to read the DiD number:**

| DiD value | What it means |
|---|---|
| **> 0** (CI excludes zero) | The treated sensor improved *more* from RL practice — it got extra benefit from PPO |
| **≈ 0** (CI includes zero) | Both sensors improved by the same amount — sensor type didn't affect how well PPO worked |
| **< 0** (CI excludes zero) | The control sensor got *more* out of RL practice |

**Concrete example:** DiD = +0.05 would mean the treated sensor's car gained an extra
5 percentage points of route completion from PPO practice, on top of whatever the control
sensor's car gained.

**The key assumption ("parallel trends"):** We assume that, without PPO fine-tuning, both
sensor setups would have improved by the same amount from BC. This is plausible because
the BC training procedure is identical for all three sensors — but it cannot be proven
from the data alone. It is a judgment call about the experimental design.

In [ ]:
did_loaded = causal_results_loaded["did"]

did_q = {"did_multi_vs_single": "Q7", "did_lidar_vs_single": "Q8", "did_lidar_vs_multi": "Q9"}

did_rows = []
for r in did_loaded:
    delta_t = r["ppo_treated_mean"] - r["bc_treated_mean"]
    delta_c = r["ppo_control_mean"] - r["bc_control_mean"]
    sig = r["ci_lower"] > 0 or r["ci_upper"] < 0
    did_rows.append({
        "Q": did_q.get(r["name"], "—"),
        "Comparison": f"{r['treated_suite']} vs {r['control_suite']}",
        "BC→PPO (treated)": f"{delta_t:+.4f}",
        "BC→PPO (control)": f"{delta_c:+.4f}",
        "DiD": f"{r['did']:+.4f}",
        "95% CI": f"[{r['ci_lower']:+.4f}, {r['ci_upper']:+.4f}]",
        "Sig.": "Yes" if sig else "No",
    })

did_df = pd.DataFrame(did_rows)
print("DiD Results — Q7 through Q9 (BC + PPO records):")
display(did_df)

print()
for r in did_loaded:
    q = did_q.get(r["name"], "?")
    delta_t = r["ppo_treated_mean"] - r["bc_treated_mean"]
    delta_c = r["ppo_control_mean"] - r["bc_control_mean"]
    sig = r["ci_lower"] > 0 or r["ci_upper"] < 0
    direction = "benefited more" if r["did"] > 0 else "benefited less"
    sig_str = "statistically significant" if sig else "not statistically significant"
    print(f"{q} | {r['name']}")
    print(f"  {r['treated_suite']} BC→PPO delta : {delta_t:+.4f}")
    print(f"  {r['control_suite']} BC→PPO delta : {delta_c:+.4f}")
    print(f"  DiD = {r['did']:+.4f}  95% CI [{r['ci_lower']:+.4f}, {r['ci_upper']:+.4f}]  — {sig_str}")
    print(f"  PPO fine-tuning {direction} for {r['treated_suite']} than {r['control_suite']}.")
    print()

In [ ]:
fig, axes = plt.subplots(1, len(did_loaded), figsize=(5 * len(did_loaded), 5), sharey=True)
if len(did_loaded) == 1:
    axes = [axes]

for ax, r in zip(axes, did_loaded):
    q = did_q.get(r["name"], "?")
    suites = [r["control_suite"], r["treated_suite"]]
    bc_means = [r["bc_control_mean"], r["bc_treated_mean"]]
    ppo_means = [r["ppo_control_mean"], r["ppo_treated_mean"]]

    x = np.array([0, 1])
    ax.plot(x, bc_means, "o--", color="coral", label="BC (pre)", linewidth=2, markersize=8)
    ax.plot(x, ppo_means, "s-", color="steelblue", label="PPO (post)", linewidth=2, markersize=8)

    # Annotate DiD
    mid_y = (ppo_means[1] + ppo_means[0]) / 2
    ax.annotate(
        f"DiD = {r['did']:+.4f}\n[{r['ci_lower']:+.4f}, {r['ci_upper']:+.4f}]",
        xy=(1.05, mid_y), fontsize=8, color="black",
        va="center", annotation_clip=False,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(suites, fontsize=9)
    ax.set_title(f"{q}: {r['treated_suite']} vs {r['control_suite']}", fontsize=10)
    ax.set_ylabel("Mean route completion")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)

fig.suptitle("DiD: BC→PPO improvement by sensor suite (Q7–Q9)", fontsize=12)
fig.tight_layout()
plt.show()

## 5.6 Sensitivity Analysis

*How much would our Q1–Q3 conclusions change if something was secretly unfair?*

### 5.6.1 The "What If We Missed Something?" Test — Rosenbaum Γ

Propensity score matching adjusts for weather and town. But what if there is some other
factor we *didn't* measure — for example, spawn-point geometry — that correlates with both
sensor type *and* route completion? Could that hidden factor explain away our finding?

The **Rosenbaum sensitivity bound** answers this with a single number: **Γ** (gamma).

> **Γ = 1.0** — any amount of hidden bias could flip the result. The finding is fragile.
>
> **Γ = 1.5** — a hidden factor would need to make one group **50% more likely** to be
> assigned the treated sensor before it could overturn the finding.
>
> **Γ = 2.0** — the hidden factor would need to **double the odds** of sensor assignment
> to flip the result. That is a very strong unmeasured confounder.

**Rule of thumb:** Γ ≥ 1.5 is considered *robust*; Γ ≤ 1.2 is considered *fragile*.

We compute Γ only for Q1–Q3 because those are the core claims of this project.
The bar chart below shows Γ for each primary sensor comparison — taller bars mean
the finding is harder to explain away.

In [ ]:
# Only primary treatments have Rosenbaum bounds
primary_results = [r for r in causal_results_loaded["psm"] if r.get("is_primary")]
treatment_names = [r["treatment"].replace("sensor_", "").replace("_vs_", " vs ") for r in primary_results]
gamma_values = [r["rosenbaum_gamma"] for r in primary_results]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(treatment_names, gamma_values, color="steelblue", edgecolor="black", alpha=0.8)

for bar, val in zip(bars, gamma_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
        f"{val:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax.axhline(y=1.0, color="red", linestyle="--", linewidth=1.5, label="Gamma = 1.0 (no hidden bias)")
ax.axhline(y=1.5, color="orange", linestyle=":", linewidth=1.5, label="Gamma = 1.5 (robust threshold)")
ax.set_ylabel("Rosenbaum Gamma")
ax.set_xlabel("Sensor Suite Comparison")
ax.set_title("Sensitivity to Unmeasured Confounding — Primary Treatments Only (Q1–Q3)")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

### 5.6.2 Did We Have Enough Comparable Drives? (Overlap Check)

Matching can only produce trustworthy results when the treated and control groups share
a common region of propensity scores — the "overlap zone." If one group's scores are all
near 1.0 while the other's are all near 0.0, matching is effectively pairing apples with
oranges, and the ATE is unreliable.

The check below reports, for each treatment:
- Whether overlap was adequate (`overlap_ok`)
- The propensity score range for each group
- The width of the overlap zone (a negative width means no overlap at all)

**If `overlap_ok = False` for a treatment:** do not use that ATE to make claims.
The fix is collecting more varied data — more test drives that span a wider range of
conditions for both the treated and control groups.

In [ ]:
for tdef in all_treatments:
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = np.array([i for i, r in enumerate(records_ppo) if t_filter(r)])
    c_idx = np.array([i for i, r in enumerate(records_ppo) if c_filter(r)])
    name = tdef["name"]

    if len(t_idx) < cfg["min_treated"] or len(c_idx) < cfg["min_treated"]:
        print(f"\n{name}: SKIPPED ({len(t_idx)} treated, {len(c_idx)} control)")
        continue

    all_idx = np.concatenate([t_idx, c_idx])
    T = np.array([1] * len(t_idx) + [0] * len(c_idx))
    X = np.column_stack([
        cov_arrays[cov][all_idx]
        for cov in tdef["covariates"]
        if cov in cov_arrays
    ])

    lr = LogisticRegression(C=1.0, max_iter=500, random_state=cfg["random_seed"])
    lr.fit(X, T)
    pscore = lr.predict_proba(X)[:, 1]
    ps_t, ps_c = pscore[T == 1], pscore[T == 0]

    overlap_range = min(ps_t.max(), ps_c.max()) - max(ps_t.min(), ps_c.min())
    frac_boundary = (np.mean(ps_t > 0.9) + np.mean(ps_c < 0.1)) / 2
    result_match = next((r for r in causal_results_loaded["psm"] if r["treatment"] == name), None)
    overlap_status = result_match["overlap_ok"] if result_match else (overlap_range >= 0 and frac_boundary < 0.5)
    tier = "primary" if tdef.get("rosenbaum") else "secondary"

    print(f"\n{name} ({tier}):")
    print(f"  Overlap OK           : {overlap_status}")
    print(f"  Treated range        : [{ps_t.min():.4f}, {ps_t.max():.4f}]")
    print(f"  Control range        : [{ps_c.min():.4f}, {ps_c.max():.4f}]")
    print(f"  Overlap width        : {overlap_range:.4f}")
    print(f"  Frac near boundary   : {frac_boundary:.4f}")

## 5.7 Limitations and Caveats

*What we can and cannot conclude — and why.*

### 5.7.1 Things We Didn't — or Couldn't — Measure

Propensity score matching adjusts for what we put into the model. Here are four things
we *didn't* control for that could still quietly bias the results:

**1. Spawn-point geometry.**
Two test drives in the same town can start on completely different roads — a straight
highway vs. a tight intersection. If one sensor group happened to get easier spawn points,
that would make it look artificially better, with nothing to do with the sensor itself.

**2. Random NPC (traffic) behavior.**
Even under the same traffic-density setting, CARLA's scripted vehicles behave differently
every run. A run where an aggressive NPC cuts across the lane is harder than one where
traffic flows smoothly — and this is random, not controlled by the experiment.

**3. GPU timing spikes.**
A frame drop during a critical moment (a tight curve, a sudden NPC brake) can cause the
AI to react late. This could systematically affect one sensor more than another if that
sensor's neural network takes longer to run.

**4. Route topology within a town.**
"Town02" covers both open highway sections and dense urban grids. Two drives coded
"Town02" may face very different numbers of intersections and curves.

The **Rosenbaum Γ** in §5.6.1 tells us how large any one of these hidden biases would
need to be before it could flip our Q1–Q3 findings.

### 5.7.2 When Should You Trust the PSM Numbers?

The PSM estimates are most reliable when all three of these are true.
Check each one before drawing policy conclusions from any treatment.

**1. Overlap is adequate** (`overlap_ok = True`, §5.6.2).
If the propensity score distributions barely touch, the matched pairs aren't truly
comparable — the treated and control drives came from too-different conditions.

**2. Post-matching balance passed** (all SMDs < 0.1, §5.4.2).
If a covariate is still imbalanced after matching, the comparison is not fair on
that dimension, and the ATE is still partially confounded.

**3. No propensity scores are clustered near 0 or 1**.
Scores near 0 mean "this drive almost certainly wouldn't be treated." Scores near 1
mean "this drive almost certainly would be treated." Matching such extreme drives to
each other produces unreliable comparisons.

If any of these conditions fail for a particular treatment, the alternatives are:
*(a)* collect more data with better coverage across conditions, or
*(b)* use a more robust estimator such as inverse-probability weighting with trimming.
For now, we flag those treatments as "interpret with caution" rather than silently
including them in conclusions.

### 5.7.3 Would These Results Hold on a Real Car?

All causal findings in this notebook come from the **CARLA simulator** — not from a
real vehicle on real roads. That matters for four reasons:

**1. CARLA's physics is simplified.**
"Hard rain" in CARLA mostly reduces camera visibility. In reality, rain also changes
braking distance, tire grip, and road reflections. The effects on a real car would
be more severe and harder to predict.

**2. Traffic is scripted.**
CARLA's other vehicles follow simple autopilot rules. Real traffic includes unpredictable
human drivers, pedestrians, cyclists, delivery riders, and construction zones — none of
which appear in these experiments.

**3. Sensor noise is approximate.**
CARLA's camera and lidar models approximate real hardware but don't reproduce all failure
modes: lens flare, moisture on the lens, lidar returns from rain droplets, or GPU latency
under load.

**4. The test routes are short and structured.**
Episodes use pre-defined segments in a small set of European-style towns. Real-world
driving involves far more variety in road type, intersection density, and unexpected events.

**Bottom line:** the *directions* of our findings (e.g., rain hurts performance, lidar
helps in low-visibility conditions, aggressive driving increases crashes) are likely to
carry over to reality. The exact *magnitudes* — how many percentage points better or
worse — are specific to this simulator and should not be treated as real-world predictions.
They are a starting point for deciding which sensors to test on real hardware, not a
substitute for real-world validation.